# Course Recommendation: Output Structure Development

## Overview

A common student request is "recommend courses similar to X." This can mean two different things depending on context:

1. **Alternatives to X** — the student hasn't taken X yet (or can't fit it in their schedule) and wants courses that cover similar material. The goal is substitution: what else covers the same topics?

2. **Follow-ons from X** — the student has already taken X, enjoyed it, and wants to go deeper. The goal is progression: what builds on what I learned?

We can differentiate the two scenarios automatically by checking whether X appears in the student's transcript. If X is in `courses_taken`, the student has completed it and wants follow-ons (Scenario 2). If not, they want alternatives (Scenario 1).

Each scenario uses a different computational model to generate recommendations, which are then injected into the LLM prompt for presentation.

---
## Scenario 1: Alternatives to X

### Computational Model

Given anchor course X (not yet taken) and the student's transcript:
1. Compute cosine similarity between X's description embedding (`all-MiniLM-L6-v2`) and all course embeddings
2. Remove courses the student has already taken
3. Check prerequisite satisfaction — split results into **takeable** (prereqs met) and **blocked** (show missing prereqs)
4. Rank by similarity score

This finds courses covering the same topics that the student could actually register for.

## Scenario 2: Follow-Ons from X

### Computational Model

Given anchor course X (already taken) and the student's transcript:
1. Find all courses that list X as a prerequisite (**direct downstream** — courses designed to follow X)
2. Compute cosine similarity between X's embedding and all course embeddings
3. Combine: downstream courses get a +0.2 score bonus over pure similarity
4. Filter by prereqs met (student must have taken X plus any other prereqs)
5. Rank by combined score

The downstream bonus ensures courses that are *designed* to follow X (listed as prereq) rank above courses that are merely topically similar. This captures the difference between "covers ML topics" and "is the next course in the ML sequence."

In [1]:
# ── Setup ──
import json, os, re, sys
import numpy as np
from src.config import BASE_MODEL, MY_MODEL, HF_TOKEN, DATA_DIR
from huggingface_hub import InferenceClient

with open(os.path.join(DATA_DIR, "courses.json"), "r") as f:
    COURSES = {c["subject_id"]: c for c in json.load(f)}

client = InferenceClient(model=MY_MODEL or BASE_MODEL, token=HF_TOKEN)

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

courses_with_desc = {cid: c for cid, c in COURSES.items()
                     if c.get("description") and len(c["description"]) > 20}
cids = list(courses_with_desc.keys())
descriptions = [f"{courses_with_desc[cid]['title']}. {courses_with_desc[cid]['description'][:500]}" for cid in cids]
embeddings = embed_model.encode(descriptions, show_progress_bar=True, batch_size=64)
cid_to_idx = {cid: i for i, cid in enumerate(cids)}

student_taken = ["18.01", "18.02", "8.01", "8.02", "5.111", "7.012",
                 "6.100A", "6.100B", "6.1010", "6.1200", "18.C06", "6.3900"]

def get_prereqs(cid):
    course = COURSES.get(cid, {})
    prereq_str = course.get("prerequisites", "")
    if not prereq_str or prereq_str.lower() == "none":
        return set()
    return set(re.findall(r'([\d]+\.[\w.]+)', prereq_str))

def get_hours(cid):
    c = COURSES.get(cid, {})
    h = (c.get("in_class_hours") or 0) + (c.get("out_of_class_hours") or 0)
    return h if h > 0 else None

def similar_courses(anchor_cid, courses_taken, top_n=7):
    if anchor_cid not in cid_to_idx:
        return [], []
    idx = cid_to_idx[anchor_cid]
    sims = cosine_similarity(embeddings[idx:idx+1], embeddings)[0]
    taken_set = set(courses_taken)
    takeable, blocked = [], []
    for i in np.argsort(sims)[::-1]:
        cid = cids[i]
        if cid == anchor_cid or cid in taken_set:
            continue
        score = float(sims[i])
        prereqs = get_prereqs(cid)
        missing = prereqs - taken_set
        if not missing:
            takeable.append((cid, score))
        else:
            blocked.append((cid, score, missing))
        if len(takeable) >= top_n and len(blocked) >= 3:
            break
    return takeable[:top_n], blocked[:3]

def get_downstream(anchor_cid):
    downstream = []
    for cid, course in COURSES.items():
        prereq_str = course.get("prerequisites", "")
        if anchor_cid in prereq_str:
            downstream.append(cid)
    return downstream

def followon_courses(anchor_cid, courses_taken, top_n=7):
    if anchor_cid not in cid_to_idx:
        return [], []
    downstream = set(get_downstream(anchor_cid))
    idx = cid_to_idx[anchor_cid]
    sims = cosine_similarity(embeddings[idx:idx+1], embeddings)[0]
    taken_set = set(courses_taken)
    all_results = []
    for i, sim_score in enumerate(sims):
        cid = cids[i]
        if cid == anchor_cid or cid in taken_set:
            continue
        is_downstream = cid in downstream
        combined = float(sim_score) + (0.2 if is_downstream else 0.0)
        prereqs = get_prereqs(cid)
        prereqs_met = prereqs.issubset(taken_set | {anchor_cid})
        missing = prereqs - taken_set - {anchor_cid}
        all_results.append((cid, combined, float(sim_score), is_downstream, prereqs_met, missing))
    all_results.sort(key=lambda x: x[1], reverse=True)
    takeable = [(c, comb, sim, ds) for c, comb, sim, ds, met, _ in all_results if met]
    blocked = [(c, comb, sim, ds, miss) for c, comb, sim, ds, met, miss in all_results if not met]
    return takeable[:top_n], blocked[:3]

print(f"Model: {MY_MODEL or BASE_MODEL}")
print(f"Courses with embeddings: {len(cids)}")
print(f"Student has taken: {', '.join(student_taken)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

Model: meta-llama/Llama-3.1-8B-Instruct
Courses with embeddings: 6800
Student has taken: 18.01, 18.02, 8.01, 8.02, 5.111, 7.012, 6.100A, 6.100B, 6.1010, 6.1200, 18.C06, 6.3900


### Scenario 1 Model Output

In [4]:
# ── Scenario 1: Similar courses to 6.3900 (if student hadn't taken it) ──
anchor = "6.3900"
title = COURSES[anchor]["title"]
# For scenario 1, pretend student hasn't taken it
taken_without = [c for c in student_taken if c != anchor]
takeable, blocked = similar_courses(anchor, taken_without)

print(f"ALTERNATIVES TO: {anchor} — {title}")
print(f"\nCan take (prereqs met):")
for cid, score in takeable:
    c = COURSES[cid]
    hrs = get_hours(cid)
    hrs_str = f"~{hrs:.0f} hrs/wk" if hrs else "hours N/A"
    print(f"  {score:.3f}  {cid:<12s} {c['title'][:50]}  ({hrs_str})")
if blocked:
    print(f"\nBlocked:")
    for cid, score, missing in blocked:
        print(f"  {score:.3f}  {cid:<12s} {COURSES[cid]['title'][:40]}  (needs: {', '.join(list(missing)[:3])})")

ALTERNATIVES TO: 6.3900 — Introduction to Machine Learning

Can take (prereqs met):
  0.682  6.862        Applied Machine Learning  (~13 hrs/wk)
  0.671  6.C01        Modeling with Machine Learning: from Algorithms to  (~9 hrs/wk)
  0.671  6.C51        Modeling with Machine Learning: from Algorithms to  (~9 hrs/wk)
  0.657  15.680       Machine Learning: Algorithms, Applications, and Co  (~13 hrs/wk)
  0.559  6.C06        Linear Algebra and Optimization  (~11 hrs/wk)
  0.552  6.9700       Studies in Artificial Intelligence and Decision Ma  (hours N/A)
  0.552  15.C571      Optimization Methods  (~11 hrs/wk)

Blocked:
  0.703  6.7900       Machine Learning  (needs: 6.3700, 18.06, 6.3800)
  0.671  6.482        Modeling with Machine Learning: from Alg  (needs: 6.0001)
  0.671  6.402        Modeling with Machine Learning: from Alg  (needs: 6.0001)


### Scenario 2 Model Output

In [5]:
# ── Scenario 2: Follow-on from 6.3900 (student has taken it) ──
anchor = "6.3900"
title = COURSES[anchor]["title"]
downstream = get_downstream(anchor)
takeable, blocked = followon_courses(anchor, student_taken)

print(f"FOLLOW-ON FROM: {anchor} — {title}")
print(f"Direct downstream: {len(downstream)} courses list {anchor} as prereq")
print(f"\nRecommended (prereqs met):")
for cid, combined, sim, is_ds in takeable:
    c = COURSES[cid]
    hrs = get_hours(cid)
    hrs_str = f"~{hrs:.0f} hrs/wk" if hrs else "hours N/A"
    tag = " ← builds on " + anchor if is_ds else ""
    print(f"  {combined:.3f}  {cid:<12s} {c['title'][:45]}  ({hrs_str}){tag}")
if blocked:
    print(f"\nBlocked:")
    for cid, combined, sim, is_ds, missing in blocked:
        print(f"  {combined:.3f}  {cid:<12s} {COURSES[cid]['title'][:40]}  (needs: {', '.join(list(missing)[:3])})")

FOLLOW-ON FROM: 6.3900 — Introduction to Machine Learning
Direct downstream: 21 courses list 6.3900 as prereq

Recommended (prereqs met):
  0.682  6.862        Applied Machine Learning  (~13 hrs/wk)
  0.671  6.C01        Modeling with Machine Learning: from Algorith  (~9 hrs/wk)
  0.671  6.C51        Modeling with Machine Learning: from Algorith  (~9 hrs/wk)
  0.657  15.680       Machine Learning: Algorithms, Applications, a  (~13 hrs/wk)
  0.605  6.8120       Tissues vs. Silicon in Machine Learning  (hours N/A) ← builds on 6.3900
  0.587  6.4210       Robotic Manipulation  (~12 hrs/wk) ← builds on 6.3900
  0.587  6.4212       Robotic Manipulation  (~12 hrs/wk) ← builds on 6.3900

Blocked:
  0.857  6.7250       Optimization for Machine Learning  (needs: 18.06)
  0.762  6.8200       Sensorimotor Learning  (needs: 6.7900)
  0.703  6.7900       Machine Learning  (needs: 18.600, 6.3700, 6.3800)


---
## Designing the LLM Output

With the computational models producing ranked results, the LLM's job is presentation. We built the following features into the prompt instructions:

- **Scenario detection:** If the anchor is in the student's transcript → follow-on prompt. If not → alternatives prompt. The LLM doesn't decide which scenario — Python does.
- **Structured sections:** For follow-ons, results are split into "courses that build directly on X" (downstream from prereq graph) and "related courses" (topically similar). For alternatives, all results are in one list.
- **Differentiated reasons:** The prompt provides course descriptions and instructs the LLM to write a one-sentence reason specific to each course, not a generic "similar to X."
- **Hours and workload:** Each course shows estimated hours/week (hidden if data unavailable, to avoid "0 hrs/wk").
- **Blocked courses as goals:** Highly similar courses the student can't take yet are shown with the specific missing prereqs, framed as "work toward these."
- **Targeted follow-up:** The turn ends with a question that helps narrow the list (e.g., "are you more interested in theory or applied work?").

Below we inject the model results into the LLM and evaluate the output.

### Test: Alternatives to 6.3900 (Scenario 1 — Turn 1)

The student hasn't taken 6.3900 and wants similar courses. Python computes similarity rankings;
the LLM presents them with differentiated reasons.

In [6]:
# ── Alternatives LLM output (Scenario 1 — student hasn't taken 6.3900) ──

anchor = "6.3900"
anchor_title = COURSES[anchor]["title"]
taken_without = [c for c in student_taken if c != anchor]
takeable_s1, blocked_s1 = similar_courses(anchor, taken_without)

context_parts_s1 = []
context_parts_s1.append(f"ALTERNATIVE COURSES TO {anchor} — {anchor_title}")
context_parts_s1.append(f"The student has NOT taken {anchor} and wants courses covering similar topics.")
context_parts_s1.append("")
context_parts_s1.append("INSTRUCTIONS:")
context_parts_s1.append("- Present all courses in a single ranked list (these are alternatives, not follow-ons).")
context_parts_s1.append("- For each course, write ONE sentence explaining what it covers and how it")
context_parts_s1.append("  relates to " + anchor + ". Use the description — don't just say 'similar to ML'.")
context_parts_s1.append("- Show hours/week where available. If hours are missing, omit the field.")
context_parts_s1.append("- Mention blocked courses briefly as future goals with their missing prereqs.")
context_parts_s1.append("- End with: 'Are you looking for something with the same theoretical depth,")
context_parts_s1.append("  or more of a hands-on / project-based approach?'")
context_parts_s1.append("")
context_parts_s1.append("RECOMMENDED ALTERNATIVES (prereqs met):\n")

for cid, score in takeable_s1:
    c = COURSES[cid]
    hrs = get_hours(cid)
    hrs_str = f", ~{hrs:.0f} hrs/wk" if hrs else ""
    desc = c.get("description", "")[:250]
    context_parts_s1.append(f"  {cid} — {c['title']} ({c.get('total_units', '?')}u{hrs_str})  Similarity: {score:.3f}")
    context_parts_s1.append(f"    Description: {desc}")
    context_parts_s1.append("")

if blocked_s1:
    context_parts_s1.append("BLOCKED (prereqs not met — future goals):\n")
    for cid, score, missing in blocked_s1:
        c = COURSES[cid]
        context_parts_s1.append(f"  {cid} — {c['title']} (needs: {', '.join(list(missing)[:3])})")
    context_parts_s1.append("")

tool_context_s1 = "\n".join(context_parts_s1)

from src.chat import SYSTEM_PROMPT
messages_s1 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": (
        f"The student asked: I can't fit {anchor} into my schedule. What other courses cover similar material?\n\n"
        f"Factual data from our system:\n---\n{tool_context_s1}\n---\n\n"
        f"Present ONLY the information above clearly and helpfully. "
        f"Do NOT suggest courses not in the data above."
    )},
]

response_s1 = client.chat_completion(messages=messages_s1, max_tokens=1000, temperature=0.3)
turn1_s1_response = response_s1.choices[0].message.content.strip()
print("=== SCENARIO 1 — TURN 1: LLM OUTPUT ===")
print(turn1_s1_response)


=== SCENARIO 1 — TURN 1: LLM OUTPUT ===
Here is the list of alternative courses to 6.3900, ranked by similarity:

1. 6.862 — Applied Machine Learning (12u, ~13 hrs/wk) 
   Description: Introduces principles, algorithms, and applications of machine learning from the point of view of modeling and prediction; formulation of learning problems; representation, over-fitting, generalization; classification, regression, reinforcement learning.

2. 15.680 — Machine Learning: Algorithms, Applications, and Computation (6u, ~13 hrs/wk) 
   Description: Develops key algorithms for machine learning with emphasis on data, applications, and computation. Includes regression, classification, support vector machines, unsupervised learning, algorithms for missing data, principal component analysis, sparse models.

3. 6.C01 — Modeling with Machine Learning: from Algorithms to Applications (6u, ~9 hrs/wk) 
   Description: Focuses on modeling with machine learning methods with an eye towards applications in 

### Test: Alternatives Turn 2 — Student says "hands-on"

The student responds to the follow-up question. The system re-ranks the same list by
similarity to "hands-on project-based applied" using embeddings.

In [7]:
# ── Scenario 1 Turn 2: Student picks "hands-on" — re-rank ──

student_reply_s1 = "I want something more hands-on and project-based."

# Re-rank alternatives by similarity to "hands-on applied projects"
applied_emb = embed_model.encode(["hands-on project-based applied machine learning practical implementation"])
applied_scores = {}
for cid, score in takeable_s1:
    if cid in cid_to_idx:
        idx = cid_to_idx[cid]
        applied_sim = float(cosine_similarity(applied_emb, embeddings[idx:idx+1])[0][0])
        applied_scores[cid] = applied_sim

# Re-rank by applied relevance
applied_ranked = sorted(takeable_s1, key=lambda x: applied_scores.get(x[0], 0), reverse=True)

context_parts_s1_t2 = []
context_parts_s1_t2.append(f"ALTERNATIVES TO {anchor} — NARROWED TO HANDS-ON / APPLIED")
context_parts_s1_t2.append(f"The student said: '{student_reply_s1}'")
context_parts_s1_t2.append("")
context_parts_s1_t2.append("INSTRUCTIONS:")
context_parts_s1_t2.append("- Present the top 3-4 courses below, re-ranked by relevance to applied/hands-on work.")
context_parts_s1_t2.append("- For each, explain what hands-on or project components it includes.")
context_parts_s1_t2.append("- If a course is more theoretical than applied, say so briefly.")
context_parts_s1_t2.append("- End with: 'Would you like to add any of these to your semester plan,")
context_parts_s1_t2.append("  or do you want more detail on a specific course?'")
context_parts_s1_t2.append("")
context_parts_s1_t2.append("APPLIED-RELEVANT COURSES (re-ranked):\n")

for cid, orig_score in applied_ranked[:5]:
    c = COURSES[cid]
    hrs = get_hours(cid)
    hrs_str = f", ~{hrs:.0f} hrs/wk" if hrs else ""
    a_score = applied_scores.get(cid, 0)
    desc = c.get("description", "")[:250]
    context_parts_s1_t2.append(f"  {cid} — {c['title']} ({c.get('total_units', '?')}u{hrs_str})")
    context_parts_s1_t2.append(f"    Applied relevance: {a_score:.3f}")
    context_parts_s1_t2.append(f"    Description: {desc}")
    context_parts_s1_t2.append("")

tool_context_s1_t2 = "\n".join(context_parts_s1_t2)

messages_s1_t2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"I can't fit {anchor} into my schedule. What other courses cover similar material?"},
    {"role": "assistant", "content": turn1_s1_response[:300] + "... [presented alternatives and asked about theoretical vs hands-on]"},
    {"role": "user", "content": (
        f"The student asked: {student_reply_s1}\n\n"
        f"Factual data from our system:\n---\n{tool_context_s1_t2}\n---\n\n"
        f"Present ONLY the information above clearly and helpfully. "
        f"Do NOT suggest courses not in the data above."
    )},
]

response_s1_t2 = client.chat_completion(messages=messages_s1_t2, max_tokens=800, temperature=0.3)
print("=== SCENARIO 1 — TURN 2: LLM OUTPUT (narrowed to applied) ===")
print(response_s1_t2.choices[0].message.content.strip())


=== SCENARIO 1 — TURN 2: LLM OUTPUT (narrowed to applied) ===
Based on your request for hands-on and project-based courses, here are the top 4 alternatives to 6.3900:

1. 6.862 — Applied Machine Learning (12u, ~13 hrs/wk)
   Applied relevance: 0.667
   Description: Introduces principles, algorithms, and applications of machine learning from the point of view of modeling and prediction; formulation of learning problems; representation, over-fitting, generalization; classification, regression, reinforcement learning. 
   Hands-on/project components: This course includes hands-on work with machine learning methods and applications.

2. 6.C01 — Modeling with Machine Learning: from Algorithms to Applications (6u, ~9 hrs/wk)
   Applied relevance: 0.508
   Description: Focuses on modeling with machine learning methods with an eye towards applications in engineering and sciences. Introduction to modern machine learning methods, from supervised to unsupervised models, with an emphasis on newer 

### Test: Follow-on from 6.3900 (Turn 1)

In [8]:
# ── Follow-on LLM output (Scenario 2 — student has taken 6.3900) ──

anchor = "6.3900"
anchor_title = COURSES[anchor]["title"]
downstream = get_downstream(anchor)
takeable, blocked = followon_courses(anchor, student_taken)

# Deduplicate similar titles (e.g., 6.4210 and 6.4212 both "Robotic Manipulation")
seen_titles = set()
deduped_takeable = []
for entry in takeable:
    cid = entry[0]
    t = COURSES[cid]["title"]
    if t not in seen_titles:
        deduped_takeable.append(entry)
        seen_titles.add(t)

context_parts = []
context_parts.append(f"FOLLOW-ON COURSES FROM {anchor} — {anchor_title}")
context_parts.append(f"The student has already taken {anchor} and wants to go deeper.")
context_parts.append("")
context_parts.append("INSTRUCTIONS:")
context_parts.append("- Split results into 'Courses that build directly on " + anchor + "' (marked below)")
context_parts.append("  and 'Related courses in the same area'.")
context_parts.append("- For each course, write ONE sentence explaining what it adds beyond " + anchor + ".")
context_parts.append("  Use the description — don't just say 'related to ML'.")
context_parts.append("- Show hours/week where available. If hours are missing, omit the field.")
context_parts.append("- Mention blocked courses briefly as future goals.")
context_parts.append("- End with: 'Are you more interested in the theoretical side, applied/project-based")
context_parts.append("  work, or a specific application area (robotics, NLP, etc.)?'")
context_parts.append("")
context_parts.append("RECOMMENDED COURSES (prereqs met):\n")

for cid, combined, sim, is_ds in deduped_takeable:
    c = COURSES[cid]
    hrs = get_hours(cid)
    hrs_str = f", ~{hrs:.0f} hrs/wk" if hrs else ""
    tag = f"  ** Builds directly on {anchor} **" if is_ds else ""
    desc = c.get("description", "")[:250]
    context_parts.append(f"  {cid} — {c['title']} ({c.get('total_units', '?')}u{hrs_str}){tag}")
    context_parts.append(f"    Description: {desc}")
    context_parts.append("")

if blocked:
    context_parts.append("BLOCKED (prereqs not met — future goals):\n")
    for cid, combined, sim, is_ds, missing in blocked:
        c = COURSES[cid]
        context_parts.append(f"  {cid} — {c['title']} (needs: {', '.join(list(missing)[:3])})")
    context_parts.append("")

tool_context = "\n".join(context_parts)

from src.chat import SYSTEM_PROMPT
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": (
        f"The student asked: I finished {anchor} and want to go deeper into machine learning. What should I take next?\n\n"
        f"Factual data from our system:\n---\n{tool_context}\n---\n\n"
        f"Present ONLY the information above clearly and helpfully. "
        f"Do NOT suggest courses not in the data above."
    )},
]

response = client.chat_completion(messages=messages, max_tokens=1000, temperature=0.3)
turn1_response = response.choices[0].message.content.strip()
print("=== TURN 1: LLM OUTPUT ===")
print(turn1_response)

=== TURN 1: LLM OUTPUT ===
**Courses that build directly on 6.3900**

1. **6.8120 — Tissues vs. Silicon in Machine Learning** (12u): Examines how brain neural circuits and function can affect the design of machine learning hardware and software, and vice versa.
2. **6.4210 — Robotic Manipulation** (15u, ~12 hrs/wk): Introduces the fundamental algorithmic approaches for creating robot systems that can autonomously manipulate physical objects in unstructured environments.

**Related courses in the same area**

1. **6.862 — Applied Machine Learning** (12u, ~13 hrs/wk): Introduces principles, algorithms, and applications of machine learning from the point of view of modeling and prediction.
2. **6.C01 — Modeling with Machine Learning: from Algorithms to Applications** (6u, ~9 hrs/wk): Focuses on modeling with machine learning methods with an eye towards applications in engineering and sciences.
3. **15.680 — Machine Learning: Algorithms, Applications, and Computation** (6u, ~13 hrs/wk): De

### Test: Follow-up (Turn 2) — Student says "theory"

The student responds to the follow-up question. The system should narrow the list to theory-focused courses, potentially re-ranking by relevance to "theory" using the embedding model.

In [ ]:
# ── Turn 2: Student picks "theory" — narrow the list ──

student_reply = "I'm more interested in the theoretical side."

# Re-rank the takeable courses by similarity to "machine learning theory"
theory_emb = embed_model.encode(["machine learning theory statistical learning theoretical foundations"])
theory_scores = {}
for cid, combined, sim, is_ds in deduped_takeable:
    if cid in cid_to_idx:
        idx = cid_to_idx[cid]
        theory_sim = float(cosine_similarity(theory_emb, embeddings[idx:idx+1])[0][0])
        theory_scores[cid] = theory_sim

# Re-rank by theory relevance
theory_ranked = sorted(deduped_takeable, key=lambda x: theory_scores.get(x[0], 0), reverse=True)

context_parts2 = []
context_parts2.append(f"FOLLOW-ON FROM {anchor} — NARROWED TO THEORY")
context_parts2.append(f"The student said: '{student_reply}'")
context_parts2.append("")
context_parts2.append("INSTRUCTIONS:")
context_parts2.append("- Present the top 3-4 courses below, re-ranked by relevance to theoretical ML.")
context_parts2.append("- For each, explain what theoretical topics it covers.")
context_parts2.append("- If a course is more applied than theoretical, say so briefly and explain")
context_parts2.append("  why it's still relevant.")
context_parts2.append("- End with a concrete next step: 'Would you like to add any of these to your")
context_parts2.append("  semester plan, or do you want more detail on a specific course?'")
context_parts2.append("")
context_parts2.append("THEORY-RELEVANT COURSES (re-ranked):\n")

for cid, combined, sim, is_ds in theory_ranked[:5]:
    c = COURSES[cid]
    hrs = get_hours(cid)
    hrs_str = f", ~{hrs:.0f} hrs/wk" if hrs else ""
    theory_score = theory_scores.get(cid, 0)
    tag = f"  [builds on {anchor}]" if is_ds else ""
    desc = c.get("description", "")[:250]
    context_parts2.append(f"  {cid} — {c['title']} ({c.get('total_units', '?')}u{hrs_str}){tag}")
    context_parts2.append(f"    Theory relevance: {theory_score:.3f}")
    context_parts2.append(f"    Description: {desc}")
    context_parts2.append("")

tool_context2 = "\n".join(context_parts2)

# Build conversation with Turn 1 history (summarized)
from src.chat import SYSTEM_PROMPT
messages2 = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"I finished {anchor} and want to go deeper into machine learning. What should I take next?"},
    {"role": "assistant", "content": turn1_response[:300] + "... [presented follow-on recommendations and asked about theory vs applied]"},
    {"role": "user", "content": (
        f"The student asked: {student_reply}\n\n"
        f"Factual data from our system:\n---\n{tool_context2}\n---\n\n"
        f"Present ONLY the information above clearly and helpfully. "
        f"Do NOT suggest courses not in the data above."
    )},
]

response2 = client.chat_completion(messages=messages2, max_tokens=800, temperature=0.3)
print("=== TURN 2: LLM OUTPUT (narrowed to theory) ===")
print(response2.choices[0].message.content.strip())